# Introduction to OpenMP

OpenMP (Open Multi-Processing) allows you to use multiple threads of the CPU when programming. It is an industry-standard API for shared-memory parallel programming, designed to help C++ developers take full advantage of multi-core processors without needing to manage low-level thread creation.

## How it works

1. The Fork-Join Model
OpenMP operates on a simple execution pattern:

- The Master Thread: The program begins as a single serial thread.
- The Fork: When a parallel directive is reached, the master thread creates a team of worker threads.
- Parallel Execution: The work is distributed across these threads.
- The Join: Once the work is finished, the threads synchronize and terminate, leaving only the master thread to continue.

2. Practical Implementation
- The most common use case is parallelising a for loop. By adding a single "pragma" line, you can distribute millions of calculations across your CPU cores.

Here is an image of how the Fork-Join model works

![fork-join](assets/fork_join.png)

In this first example function we can see how to print Hello World normally. There is only standard c++ syntax.

In [1]:
#include <iostream>
#include <omp.h>

In [2]:
void example1() {
  std::cout << "Hello World!" << std::endl;
}
example1();

Hello World!


Now, let's use OpenMP to run the same code in parallel. By adding a simple compiler directive, we tell the system to create a "team" of threads.

```#pragma omp parallel```  This directive tells the compiler to execute the following block of code in parallel using multiple threads.

The result is that instead of printing once, you will see "Hello World!" printed multiple times—once for every available logical core on your CPU.

In [3]:
void example2() {
    #pragma omp parallel
    {
    std::cout << "Hello World!" << std::endl;
    }
}
example2();

Hello World!
Hello World!
Hello World!
Hello World!
Hello World!
Hello World!
Hello World!Hello World!



There is one thing we need to address before we continue

When we use #pragma omp parallel, OpenMP spawns multiple threads that all execute the code inside the curly braces simultaneously. Every thread is trying to talk to our monitor at the exact same time.

Thread A might finish sending "Hello World!" but, before it can send the newline, Thread B jumps in and prints its own "Hello World!". This results in two greetings on one line, followed by two newlines later.

We fix this by using this by using ```#pragma omp critical```. It makes each thread wait for each other. This significantly slows down the program and loses the whole point of using multiple threads, but we are going to use it here in some examples to show you better how OpenMP works. Below we have a function that uses this fix.

In [4]:
void example3() {
    #pragma omp parallel
    {
        #pragma omp critical
        {
            std::cout << "Hello World! (" << omp_get_thread_num() << ")" << std::endl;
        }
    }
}
example3();

Hello World! (0)
Hello World! (5)
Hello World! (6)
Hello World! (4)
Hello World! (1)
Hello World! (3)
Hello World! (7)
Hello World! (2)


Other things we can do in OMP, includes using a set number of threads that are going to be used: ```#pragma omp parallel num_threads(2)``` 

In [5]:
void example4() {
    #pragma omp parallel
    {
        #pragma omp critical
        {
            std::cout << "Hello World! (" << omp_get_thread_num() << ")" << std::endl;
        }
    }

    std::cout << "This is another message! (" << omp_get_thread_num() << ")" << std::endl;

    #pragma omp parallel num_threads(2)
    {
    std::cout << "Goodbye World! (" << omp_get_thread_num() << ")" << std::endl;
    }
}
example4();

Hello World! (0)
Hello World! (6)
Hello World! (5)
Hello World! (3)
Hello World! (4)
Hello World! (1)
Hello World! (7)
Hello World! (2)
This is another message! (0)
Goodbye World! (Goodbye World! (01)
)


Now that you know basic syntax we can do a speed benchmark. We are going to time filling two arrays with 1 million elements each, adding them and calculating the average. In the average calculation there is a problem that can occur. In parallel programming, if multiple threads try to add to the same average variable at the same time, they will overwrite each other. When we use ```reduction(+:average)```, OpenMP does the following:

- Each thread gets its own private mini-average variable initialized to 0.
- Each thread calculates the sum for its assigned chunk of the array (e.g., Thread 1 sums indices 0 to 1000, Thread 2 sums 1001 to 2000).
- Once all threads finish, OpenMP safely adds all those private mini-sums together into the final global average variable.

In [6]:
void example5() {
    double start_time = omp_get_wtime();
    double start_loop;
    
    const int N = 1000000000;
    int* a = new int[N];
    int* b = new int[N];
    
    start_loop = omp_get_wtime();
    #pragma omp parallel for
    for (int i=0; i<N; i++) {
        a[i] = 1.0;
    }
    std::cout << "Initialize a[] time: " << omp_get_wtime()-start_loop << std::endl;

    start_loop = omp_get_wtime();
    #pragma omp parallel for
    for (int i=0; i<N; i++) {
        b[i] = 1.0 + double(i);
    }
    std::cout << "Initialize b[] time: " << omp_get_wtime()-start_loop << std::endl;

    start_loop = omp_get_wtime();
    #pragma omp parallel for
    for (int i=0; i<N; i++) {
        a[i] = a[i] + b[i];
    }
    std::cout << "Add arrays time: " << omp_get_wtime()-start_loop << std::endl;
    
    start_loop = omp_get_wtime();
    double average = 0.0;
    #pragma omp parallel for reduction(+:average)
    for (int i=0; i<N; i++) {
        average += a[i];
    }
    average = average/double(N);
    std::cout << "Average result time: " << omp_get_wtime()-start_loop << std::endl;
    
    std::cout << "Average: " << average << std::endl;

    std::cout << "Total time: " << omp_get_wtime()-start_time << std::endl;
}
example5();

Initialize a[] time: 0.657814
Initialize b[] time: 0.856986
Add arrays time: 0.581521
Average result time: 0.449791
Average: 5e+08
Total time: 2.54675
